<a href="https://colab.research.google.com/github/peteparker123/Cache_Augmented_Generation/blob/main/CAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
from transformers import (
    AutoTokenizer,
    #BitsAndBytesConfig,
    AutoModelForCausalLM)

#import bitsandbytes as bnb
from transformers.cache_utils import DynamicCache

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [4]:
##bnb_config = BitsAndBytesConfig(
 #   load_in_4bit=True,
 #   bnb_4bit_use_double_quant=True,
 #   bnb_4bit_quant_type="nf4",
 #   bnb_4bit_compute_dtype=torch.bfloat16)

model_id  = "meta-llama/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model     = AutoModelForCausalLM.from_pretrained(
            model_id,
            #quantization_config=bnb_config,
            device_map='auto')

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [10]:
!pip install langchain_community pypdf

In [11]:
from langchain_community.document_loaders import PyPDFLoader


# ============================================================
# LOAD PDF
# ============================================================

pdf_path = "/content/resume_finale.pdf"

loader = PyPDFLoader(pdf_path)

documents = loader.load()

print(f"Loaded {len(documents)} pages")


# ============================================================
# CONVERT LANGCHAIN DOCUMENTS TO STRING
# ============================================================

knowledge = "\n\n".join(
    document.page_content
    for document in documents
)

print(f"Knowledge characters: {len(knowledge)}")

/tmp/ipykernel_459/3947855875.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 1 pages
Knowledge characters: 2125


In [12]:
import torch
from transformers.cache_utils import DynamicCache


# ============================================================
# 1. PREPROCESS KNOWLEDGE AND CREATE KV CACHE
# ============================================================

def preprocess_knowledge(
    model,
    tokenizer,
    prompt: str
) -> DynamicCache:

    # Get the device where the model's embedding layer is stored
    embed_device = model.model.embed_tokens.weight.device

    # Convert knowledge prompt into token IDs
    input_ids = tokenizer.encode(
        prompt,
        return_tensors="pt"
    ).to(embed_device)

    # Create an empty DynamicCache
    past_key_values = DynamicCache()

    # Inference only — no gradients needed
    with torch.no_grad():

        outputs = model(
            input_ids=input_ids,
            past_key_values=past_key_values,
            use_cache=True,
            output_attentions=False,
            output_hidden_states=False
        )

    # Return the populated KV cache
    return outputs.past_key_values


# ============================================================
# 2. PREPARE KNOWLEDGE KV CACHE
# ============================================================

def prepare_kvcache(
    documents,
    answer_instruction: str = None
):

    if answer_instruction is None:
        answer_instruction = (
            "Answer questions using only the provided resume. "
            "Give accurate, concise, and direct answers. "
            "Do not provide information that is not present in the resume."
        )

    knowledges = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an AI resume analyzer.

Your task is to analyze the provided resume and answer questions about the candidate.

You can answer questions about:
- Candidate information
- Education
- Technical skills
- Programming languages
- Frameworks and tools
- Projects
- Work experience
- Internships
- Certifications
- Achievements
- Research experience
- Areas of expertise

Follow these rules:
1. Answer questions using only information available in the provided resume.
2. Do not invent or assume information that is not present in the resume.
3. If the requested information is not available in the resume, clearly say that the information is not mentioned.
4. Give concise and direct answers unless the user requests a detailed explanation.
5. When asked about projects, skills, education, or experience, provide accurate information extracted from the resume.
6. Do not include unrelated information.

<|eot_id|><|start_header_id|>user<|end_header_id|>

The candidate's resume is provided below.

---------------- RESUME ----------------

{documents}

----------------------------------------

{answer_instruction}

Question:
"""

    # Create knowledge KV cache
    kv = preprocess_knowledge(
        model,
        tokenizer,
        knowledges
    )

    # Get original knowledge cache length
    kv_len = kv.get_seq_length()

    print("KV length:", kv_len)

    return kv, kv_len


# ============================================================
# 3. CLEAN / RESTORE KV CACHE
# ============================================================

def clean_up(
    kv: DynamicCache,
    origin_len: int
):

    """
    Remove query + generated answer KV states
    and restore the cache to the original
    knowledge-only length.
    """

    kv.crop(origin_len)


# ============================================================
# 4. CUSTOM GREEDY GENERATION
# ============================================================

def generate(
    model,
    tokenizer,
    input_ids: torch.Tensor,
    past_key_values,
    max_new_tokens: int = 100
) -> torch.Tensor:

    embed_device = model.model.embed_tokens.weight.device

    input_ids = input_ids.to(embed_device)

    # Save query length so we can return only generated tokens
    input_length = input_ids.shape[-1]

    # Initially output contains the query
    output_ids = input_ids.clone()

    # First model call processes the ENTIRE query
    next_token = input_ids

    # Get EOS token
    eos_token_id = tokenizer.eos_token_id

    # Get Llama end-of-turn token
    eot_token_id = tokenizer.convert_tokens_to_ids(
        "<|eot_id|>"
    )

    # All tokens that should stop generation
    stop_token_ids = {
        token_id
        for token_id in [
            eos_token_id,
            eot_token_id
        ]
        if token_id is not None
    }

    with torch.no_grad():

        for _ in range(max_new_tokens):

            # First iteration:
            #
            # Knowledge KV Cache
            # +
            # Entire Query
            #
            # Later iterations:
            #
            # Updated KV Cache
            # +
            # One Generated Token

            outputs = model(
                input_ids=next_token,
                past_key_values=past_key_values,
                use_cache=True
            )

            # Get logits for the final token position
            next_token_logits = outputs.logits[:, -1, :]

            # Greedy decoding:
            # select token with highest logit
            next_token = next_token_logits.argmax(
                dim=-1
            ).unsqueeze(-1)

            # Update KV cache
            past_key_values = outputs.past_key_values

            # Add generated token to output
            output_ids = torch.cat(
                [output_ids, next_token],
                dim=1
            )

            # Stop at EOS or EOT
            if next_token.item() in stop_token_ids:
                break

    # Return ONLY newly generated answer tokens
    return output_ids[:, input_length:]

# ============================================================
# CREATE KNOWLEDGE KV CACHE
# ============================================================

knowledge_cache, kv_len = prepare_kvcache(
    documents=knowledge,
    answer_instruction="""
Answer questions using only the provided resume.
Give accurate and direct answers.
Do not invent information that is not present in the resume.
If the requested information is unavailable, say that it is not mentioned in the resume.
"""
)

KV length: 771


In [14]:
# ============================================================
# 6. USER QUERY
# ============================================================

query = "What experience does the candidate have with Large Language Models?"

# ============================================================
# 7. FORMAT QUERY
# ============================================================

# Complete the existing user message,
# close the user turn,
# and start the assistant turn.

query_prompt = f"""{query}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""


# ============================================================
# 8. RESTORE ORIGINAL KNOWLEDGE CACHE
# ============================================================

clean_up(
    knowledge_cache,
    kv_len
)


# ============================================================
# 9. TOKENIZE QUERY
# ============================================================

embed_device = model.model.embed_tokens.weight.device

input_ids = tokenizer.encode(
    query_prompt,
    return_tensors="pt"
).to(embed_device)


# ============================================================
# 10. GENERATE ANSWER
# ============================================================

output = generate(
    model=model,
    tokenizer=tokenizer,
    input_ids=input_ids,
    past_key_values=knowledge_cache,
    max_new_tokens=700
)


# ============================================================
# 11. DECODE ANSWER
# ============================================================

generated_text = tokenizer.decode(
    output[0],
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)


# ============================================================
# 12. PRINT RESULT
# ============================================================

print(
    f"Response of the model:\n{generated_text}"
)

Response of the model:
The candidate has hands-on experience developing transformer-based models, including a 40M-parameter Tamil transformer language model, and has worked with LLM pipelines, specifically with LangChain, LlamaIndex, FAISS, and ChromaDB.


In [15]:
# ============================================================
# 6. USER QUERY
# ============================================================

query = "What projects has the candidate built?"

# ============================================================
# 7. FORMAT QUERY
# ============================================================

# Complete the existing user message,
# close the user turn,
# and start the assistant turn.

query_prompt = f"""{query}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""


# ============================================================
# 8. RESTORE ORIGINAL KNOWLEDGE CACHE
# ============================================================

clean_up(
    knowledge_cache,
    kv_len
)


# ============================================================
# 9. TOKENIZE QUERY
# ============================================================

embed_device = model.model.embed_tokens.weight.device

input_ids = tokenizer.encode(
    query_prompt,
    return_tensors="pt"
).to(embed_device)


# ============================================================
# 10. GENERATE ANSWER
# ============================================================

output = generate(
    model=model,
    tokenizer=tokenizer,
    input_ids=input_ids,
    past_key_values=knowledge_cache,
    max_new_tokens=700
)


# ============================================================
# 11. DECODE ANSWER
# ============================================================

generated_text = tokenizer.decode(
    output[0],
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)


# ============================================================
# 12. PRINT RESULT
# ============================================================

print(
    f"Response of the model:\n{generated_text}"
)

Response of the model:
The candidate has built the following projects:

1. Tamil GPT (40M Parameters)
2. Legal AI Chatbot (LangChain)
3. ClassMind – Real-Time AI Notes
4. Chat Epstein – RAG Agent (Ongoing)


In [16]:
# ============================================================
# 6. USER QUERY
# ============================================================

query = "What is the github url link?"

# ============================================================
# 7. FORMAT QUERY
# ============================================================

# Complete the existing user message,
# close the user turn,
# and start the assistant turn.

query_prompt = f"""{query}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""


# ============================================================
# 8. RESTORE ORIGINAL KNOWLEDGE CACHE
# ============================================================

clean_up(
    knowledge_cache,
    kv_len
)


# ============================================================
# 9. TOKENIZE QUERY
# ============================================================

embed_device = model.model.embed_tokens.weight.device

input_ids = tokenizer.encode(
    query_prompt,
    return_tensors="pt"
).to(embed_device)


# ============================================================
# 10. GENERATE ANSWER
# ============================================================

output = generate(
    model=model,
    tokenizer=tokenizer,
    input_ids=input_ids,
    past_key_values=knowledge_cache,
    max_new_tokens=700
)


# ============================================================
# 11. DECODE ANSWER
# ============================================================

generated_text = tokenizer.decode(
    output[0],
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)


# ============================================================
# 12. PRINT RESULT
# ============================================================

print(
    f"Response of the model:\n{generated_text}"
)

Response of the model:
The candidate's GitHub URL link is github.com/peteparker123.
